In [ ]:
macro_data_path="../../../datasets/macro"
# read files from macro_data_path
import os
import pandas as pd
import numpy as np
# some files are csv and some are txt 
# read all files in the directory
def read_files_from_path(path):
    files = os.listdir(path)
    data = {}
    for file in files:
        print(file)
        if file.endswith(".csv"):
            data[file] = pd.read_csv(os.path.join(path, file))
        elif file.endswith(".txt"):
            data[file] = pd.read_table(os.path.join(path, file))
    return data
# print the first 5 rows of each file
data = read_files_from_path(macro_data_path)
for file in data:
    print(file)
    print(data[file].head(1))
    print("\n\n")

In [ ]:
# unify the column names
# change "date","DATE","period" to "date"

for file in data:
    if "date" in data[file].columns:
        data[file] = data[file].rename(columns={"date":"date"})
    elif "DATE" in data[file].columns:
        data[file] = data[file].rename(columns={"DATE":"date"})
    elif "period" in data[file].columns:
        data[file] = data[file].rename(columns={"period":"date"})
    else:
        print("no date column in {}".format(file))

import pandas as pd

# List of possible date formats
date_formats = [
    "%Y-%m-%d",    # 1999-3-31, 2000-01-01
    "%d-%m-%Y",    # 05-12-2021
    "%Ym%m",       # 1976m8
]

# Iterate through the files
for file in data:
    if "date" in data[file].columns:
        date_column = data[file]["date"]
        converted_date = None
        print(f"Converting dates in {file}")
        print("date before conversion",date_column.head(5))

        # Try each format
        for fmt in date_formats:
            try:
                converted_date = pd.to_datetime(date_column, format=fmt, errors='coerce')
                # Break if parsing succeeds (i.e., no more NaT values)
                if not converted_date.isna().all():
                    break
            except Exception as e:
                continue

        # Fallback to automatic parsing for unhandled formats
        if converted_date is None or converted_date.isna().any():
            try:
                converted_date = pd.to_datetime(date_column, errors='coerce')
            except Exception as e:
                print(f"Error parsing dates in file {file}: {e}")

        # Check for remaining NaT values and log them
        if converted_date.isna().any():
            print(f"Date format error in {file}: ", data[file].loc[converted_date.isna(), "date"].values)

        # Retain only month and year
        data[file]["date"] = converted_date.dt.to_period('M').dt.to_timestamp()
        print(f"Converted dates in {file} to {converted_date.dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')}")

    else:
        print(f"No date column in {file}")

    

In [ ]:
# cast all non-date columns to numeric
for file in data:
    for column in data[file].columns:
        if column != "date":
            data[file][column] = pd.to_numeric(data[file][column], errors='coerce')

In [ ]:
# group mean by date
for file in data:
    data[file] = data[file].groupby("date").mean()
    # print(data[file].head(5))
    # still use date as a column
    data[file].reset_index(inplace=True)

In [ ]:
# merge all data by date column
merged_data = None
for file in data:
    if merged_data is None:
        merged_data = data[file]
    else:
        merged_data = pd.merge(merged_data, data[file], on="date", how="outer")

In [ ]:
# select date range from 2000-01 to 2024-12
merged_data = merged_data[(merged_data["date"] >= "2000-01") & (merged_data["date"] <= "2024-12")]

In [ ]:
# describe the data, check nan values of each column
# forward fill nan values, then backfill for any leading NaNs
merged_data = merged_data.sort_values("date")
merged_data = merged_data.ffill()
merged_data = merged_data.bfill()
print(merged_data.describe())

In [ ]:
# save the merged data to a csv file
data_folder = "../../../datasets/macro_processed"
# change the 'date' column to "Date"
merged_data = merged_data.rename(columns={"date":"Date"})
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
merged_data.to_csv(os.path.join(data_folder, "macro_data.csv"), index=False)

In [ ]:
#read all stock data from the folder
import os
import pandas as pd
# Single-source Tiingo EOD (split + dividend adjusted). Replaces the prior
# Yahoo(<2016) + Alpaca(>=2016) merge which introduced a spurious return
# spike on 2016-01-04 (adjustment convention mismatch between sources).
stock_data_path = "../../../workdir/tiingo_day_prices_dj30"
# Skip the downloader log file, only load per-ticker CSVs
def read_files_from_path(path):
    files = os.listdir(path)
    data = {}
    for file in files:
        if not file.endswith(".csv"):
            continue
        filepath = os.path.join(path, file)
        if not os.path.isfile(filepath):
            continue
        file_name = file.split(".")[0]
        try:
            df = pd.read_csv(filepath)
        except pd.errors.EmptyDataError:
            print(f"Skipping empty file: {file}")
            continue
        if len(df) == 0:
            print(f"Skipping file with no rows: {file}")
            continue
        # Tiingo per-ticker CSVs already carry a 'ticker' column; the
        # downloader log CSV (if any) would not have OHLCV columns.
        required = {"Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"}
        if not required.issubset(df.columns):
            print(f"Skipping non-OHLCV file: {file}")
            continue
        data[file_name] = df
    return data
data = read_files_from_path(stock_data_path)
print(f"Loaded {len(data)} tickers: {sorted(data.keys())}")
# concatenate all stock data together into one dataframe
stock_data = None
for file in data:
    ticker = file.split(".")[0]
    df = data[file].copy()
    # Enforce ticker column matches filename (defensive; Tiingo already sets it)
    df["ticker"] = ticker
    if stock_data is None:
        stock_data = df
    else:
        stock_data = pd.concat([stock_data, df], axis=0)
unique_dates = stock_data["Date"].unique()
print("unique dates", len(unique_dates))
# drop the row where Date is NaN
stock_data = stock_data.dropna(subset=["Date"])
print(stock_data.columns)
print(stock_data.head(5))
print("date range of stock data", stock_data["Date"].min(), stock_data["Date"].max())
print("size of stock data", stock_data.shape)
print("unique tickers", stock_data["ticker"].unique())
# print the NaN numb of each column
print(stock_data.isna().sum())
# save the merged stock data to a csv file
stock_data = stock_data.sort_values(["ticker", "Date"]).reset_index(drop=True)
data_folder = "../../../datasets/stock_data"
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
# Write the canonical filename used by downstream notebooks. Also keep the
# legacy misspelled name as a symlink-style copy for backwards compat.
stock_data.to_csv(os.path.join(data_folder, "merged_stock_data.csv"), index=False)
#stock_data.to_csv(os.path.join(data_folder, "megred_stock_data.csv"), index=False)


In [ ]:
import pandas as pd
# merge the stock data with the macro data
stock_data_path="../../../datasets/stock_data/merged_stock_data.csv"
selected_macro_path = "macro_list.txt"
with open(selected_macro_path, "r") as f:
    Macro_features = f.read().splitlines()
macro_data_path="../../../datasets/macro_processed/macro_data.csv"
# merge the stock data with the macro data
stock_data = pd.read_csv(stock_data_path)
full_macro_data = pd.read_csv(macro_data_path)
selected_macro_data = full_macro_data[['Date']+Macro_features]
print("date stamps of macro data",selected_macro_data.head(10))
# now selected_macro_data only have the first day of each month, we need to forward fill the data to all days in the month
selected_macro_data["Date"] = pd.to_datetime(selected_macro_data["Date"])
selected_macro_data.set_index("Date", inplace=True)
selected_macro_data = selected_macro_data.resample("D").ffill()
print("resampled macro data",selected_macro_data.head(10))
# save the resampled macro data to a csv file
data_folder = "../../../datasets/macro_processed"
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
selected_macro_data.to_csv(os.path.join(data_folder, "macro_data_resampled.csv"))